# Simulation Analysis

This notebook analyses the repeat-level simulation results used to compare
standard BART variable inclusion proportions with the log
marginal-likelihood-weighted importance measure.

The notebook does **not** rerun the BART simulation experiments. The committed
CSV files in `results/` contain the expensive experiment outputs generated by
the scripts in `Experiments/`. This notebook validates those outputs,
aggregates the repeat-level results, and produces the derived tables stored in
`results/export_tables/`.

The accompanying report contains the full methodology and interpretation of
the experiments.

## Setup and experiment validation

In [1]:
from pathlib import Path

import pandas as pd

from scenario_analysis_helpers import (
    aggregate_experiment_results,
    display_summary_table,
    display_n_selected_summary,
)

from sensitivity_analysis_helpers import (
    aggregate_hyperparameter_results,
    display_hyperparameter_summary,
    aggregate_paired_hyperparameter_results,
    display_paired_summary,
    aggregate_hyperparameter_sensitivity,
    display_sensitivity_comparison,
    plot_hyperparameter_sensitivity
)

In [2]:
RESULTS_DIR = Path("results")
HYPERPARAM_DIR = RESULTS_DIR / "hyperparams_results"
EXPORT_DIR = RESULTS_DIR / "export_tables"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
MAIN_EXPERIMENTS = {
    "cc1": {
        "file": "cc1.csv",
        "n": {500},
        "p": {50, 200},
        "s2": {1.0},
        "repeats": 100,
        "n_true": 5,
    },
    "cc2": {
        "file": "cc2.csv",
        "n": {500},
        "p": {50, 200},
        "s2": {1.0},
        "repeats": 100,
        "n_true": 4,
    },
    "cm1": {
        "file": "cm1.csv",
        "n": {500, 1000},
        "p": {50, 200},
        "s2": {1.0, 10.0},
        "repeats": 100,
        "n_true": 5,
    },
    "cm2": {
        "file": "cm2.csv",
        "n": {500, 1000},
        "p": {84},
        "s2": {1.0, 10.0},
        "repeats": 100,
        "n_true": 6,
    },
    "bm1": {
        "file": "bm1.csv",
        "n": {500, 1000},
        "p": {50, 200},
        "s2": {1.0},
        "repeats": 100,
        "n_true": 5,
    },
    "bm2": {
        "file": "bm2.csv",
        "n": {500, 1000},
        "p": {84},
        "s2": {1.0},
        "repeats": 100,
        "n_true": 6,
    },
}

In [4]:
SENSITIVITY_EXPERIMENTS = {
    "cc1": {
        "file": "cc1.csv",
        "n": 500,
        "p": 50,
        "s2": 1.0,
        "repeats": 50,
        "label": "Continuous predictors",
    },
    "cm2": {
        "file": "cm2.csv",
        "n": 500,
        "p": 84,
        "s2": 10.0,
        "repeats": 50,
        "label": "Mixed predictors",
    },
    "bm1": {
        "file": "bm1.csv",
        "n": 1000,
        "p": 50,
        "s2": 1.0,
        "repeats": 50,
        "label": "Binary response",
    },
}

In [5]:
EXPECTED_HYPERPARAMETER_CONFIGS = {
    "reference": {
        "varied_parameter": "reference",
        "m": 20,
        "alpha": 0.95,
        "beta": 2.0,
        "k": 2.0,
    },
    "m_low": {
        "varied_parameter": "m",
        "m": 10,
        "alpha": 0.95,
        "beta": 2.0,
        "k": 2.0,
    },
    "m_high": {
        "varied_parameter": "m",
        "m": 50,
        "alpha": 0.95,
        "beta": 2.0,
        "k": 2.0,
    },
    "beta_low": {
        "varied_parameter": "beta",
        "m": 20,
        "alpha": 0.95,
        "beta": 1.0,
        "k": 2.0,
    },
    "beta_high": {
        "varied_parameter": "beta",
        "m": 20,
        "alpha": 0.95,
        "beta": 3.0,
        "k": 2.0,
    },
    "k_low": {
        "varied_parameter": "k",
        "m": 20,
        "alpha": 0.95,
        "beta": 2.0,
        "k": 1.0,
    },
    "k_high": {
        "varied_parameter": "k",
        "m": 20,
        "alpha": 0.95,
        "beta": 2.0,
        "k": 3.0,
    },
}

In [6]:
def load_and_validate_main_result(name, spec):
    df = pd.read_csv(RESULTS_DIR / spec["file"])

    if set(df["scenario"]) != {name}:
        raise ValueError(f"Unexpected scenario values in {spec['file']}.")

    if not df["status"].eq("complete").all():
        raise ValueError(f"{spec['file']} contains failed experiment rows.")

    for column in ("n", "p", "s2"):
        observed = set(df[column].unique())
        expected = spec[column]

        if observed != expected:
            raise ValueError(
                f"{name}: expected {column}={expected}, got {observed}."
            )

    counts = df.groupby(["n", "p", "s2"]).size()

    if not counts.eq(spec["repeats"]).all():
        raise ValueError(
            f"{name}: expected {spec['repeats']} repeats per configuration."
        )

    return df

In [7]:
def load_and_validate_sensitivity_result(name, spec):
    path = HYPERPARAM_DIR / spec["file"]
    df = pd.read_csv(path)

    if set(df["scenario"]) != {name}:
        raise ValueError(
            f"{path} contains unexpected scenario values."
        )

    if not df["status"].eq("complete").all():
        failed = df.loc[
            ~df["status"].eq("complete"),
            ["repeat", "config_id", "error"],
        ]
        raise ValueError(
            f"{path} contains incomplete experiment rows:\n{failed}"
        )

    expected_scalar_values = {
        "n": spec["n"],
        "p": spec["p"],
        "s2": spec["s2"],
    }

    for column, expected in expected_scalar_values.items():
        observed = set(df[column].unique())

        if observed != {expected}:
            raise ValueError(
                f"{name}: expected {column}={expected}, "
                f"got {sorted(observed)}."
            )

    expected_configs = set(EXPECTED_HYPERPARAMETER_CONFIGS)

    observed_configs = set(df["config_id"].unique())

    if observed_configs != expected_configs:
        raise ValueError(
            f"{name}: unexpected configuration set. "
            f"Expected {sorted(expected_configs)}, "
            f"got {sorted(observed_configs)}."
        )

    repeat_ids = sorted(df["repeat"].unique())

    if len(repeat_ids) != spec["repeats"]:
        raise ValueError(
            f"{name}: expected {spec['repeats']} repeats, "
            f"got {len(repeat_ids)}."
        )

    # Every repeat must contain every configuration exactly once.
    configuration_counts = (
        df.groupby(["repeat", "config_id"])
        .size()
        .unstack(fill_value=0)
    )

    configuration_counts = configuration_counts.reindex(
        columns=sorted(expected_configs),
        fill_value=0,
    )

    if not configuration_counts.eq(1).all().all():
        raise ValueError(
            f"{name}: each repeat must contain exactly one row "
            "for each hyperparameter configuration."
        )

    # All seven configurations within a repeat must share the same seed.
    seeds_per_repeat = df.groupby("repeat")["seed"].nunique()

    if not seeds_per_repeat.eq(1).all():
        raise ValueError(
            f"{name}: configurations within a repeat do not "
            "share the same seed."
        )

    # Check the actual hyperparameter values attached to each config_id.
    for config_id, expected in EXPECTED_HYPERPARAMETER_CONFIGS.items():
        d = df[df["config_id"].eq(config_id)]

        for column, expected_value in expected.items():
            observed = set(d[column].unique())

            if observed != {expected_value}:
                raise ValueError(
                    f"{name}/{config_id}: expected "
                    f"{column}={expected_value}, got {observed}."
                )

    return df

In [8]:
main_results = {
    name: load_and_validate_main_result(name, spec)
    for name, spec in MAIN_EXPERIMENTS.items()
}

In [9]:
sensitivity_results = {
    name: load_and_validate_sensitivity_result(name, spec)
    for name, spec in SENSITIVITY_EXPERIMENTS.items()
}

In [10]:
pd.DataFrame(
    [
        {
            "Scenario": name.upper(),
            "Rows": len(df),
            "n": sorted(df["n"].unique()),
            "p": sorted(df["p"].unique()),
            "sigma2": sorted(df["s2"].unique()),
            "Repeats": spec["repeats"],
        }
        for (name, spec), df in zip(
            MAIN_EXPERIMENTS.items(),
            main_results.values(),
        )
    ]
)

,Scenario,Rows,n,p,sigma2,Repeats
0,CC1,200,[500],"[50, 200]",[1.0],100
1,CC2,200,[500],"[50, 200]",[1.0],100
2,CM1,800,"[500, 1000]","[50, 200]","[1.0, 10.0]",100
3,CM2,400,"[500, 1000]",[84],"[1.0, 10.0]",100
4,BM1,400,"[500, 1000]","[50, 200]",[1.0],100
5,BM2,200,"[500, 1000]",[84],[1.0],100


In [11]:
pd.DataFrame(
    [
        {
            "Scenario": name.upper(),
            "Setting": spec["label"],
            "n": spec["n"],
            "p": spec["p"],
            "sigma2": spec["s2"],
            "Repeats": spec["repeats"],
            "Configurations": df["config_id"].nunique(),
        }
        for name, spec in SENSITIVITY_EXPERIMENTS.items()
        for df in [sensitivity_results[name]]
    ]
)

,Scenario,Setting,n,p,sigma2,Repeats,Configurations
0,CC1,Continuous predictors,500,50,1.0,50,7
1,CM2,Mixed predictors,500,84,10.0,50,7
2,BM1,Binary response,1000,50,1.0,50,7


## Main simulation study

The primary comparison uses the global-SE permutation threshold. The raw CSV
files also contain local and global-max results, but these are not repeated
throughout the notebook because the final report treats global-SE as the
primary feature-selection rule.

### CC1 - Continuous predictors, continuous response

In the first scenario the Friedman function was used, defined as:
$$
    f(x) = 10 \sin{(\pi x_1 x_2)} + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5,
$$
with $x_1, \dots, x_p \sim Unif(0, 1)$ independent.

In [12]:
cc1_summary, cc1_selected = aggregate_experiment_results(
    main_results["cc1"]
)

display_summary_table(
    cc1_summary,
    "global_se",
    EXPORT_DIR / "cc1_summary.csv",
)

display_n_selected_summary(
    cc1_selected,
    MAIN_EXPERIMENTS["cc1"]["n_true"],
    EXPORT_DIR / "cc1_selected.csv",
)

#### Global-SE threshold

#### Number of selected variables

### CC2 - Continuous predictors, continuous response

For the second continuous-continuous scenario, detection for correlated covariates was compared.

Here $x_1,\dots,x_p$ came from a multivariate normal distribution $N(0, \Sigma)$ where $\Sigma_{ij} = 0.3^{|i - j|}$ and $f$ is defined as:
$$
    f(x) = 2 x_1 x_4 + 2 x_7 x_{10}

$$

In [13]:
cc2_summary, cc2_selected = aggregate_experiment_results(
    main_results["cc2"]
)

display_summary_table(
    cc2_summary,
    "global_se",
    EXPORT_DIR / "cc2_summary.csv",
)

display_n_selected_summary(
    cc2_selected,
    MAIN_EXPERIMENTS["cc2"]["n_true"],
    EXPORT_DIR / "cc2_selected.csv",
)

#### Global-SE threshold

#### Number of selected variables

### CM1 - Mixed predictors, continuous response

In the first scenario, predictors $x_1, \dots, x_{\lceil p/2 \rceil} \sim Bernoulli(0.5)$ independently and $x_{\lceil p/2 \rceil + 1}, \dots, x_p \sim Unif(0, 1)$ independently. Function $f$ is:
$$
    f(x) = 10 \sin{(\pi x_{\lceil p/2 \rceil +1} x_{\lceil p/2 \rceil +2})} + 20(x_{\lceil p/2 \rceil +3} - 0.5)^2 + 10x_1 + 5x_2
$$

Here we consider $n \in \{ 500, 1000 \}$, $p \in \{ 50, 200 \}$ and $\sigma^2 \in \{1, 10\}$.

In [14]:
cm1_summary, cm1_selected = aggregate_experiment_results(
    main_results["cm1"]
)

display_summary_table(
    cm1_summary,
    "global_se",
    EXPORT_DIR / "cm1_summary.csv",
)

display_n_selected_summary(
    cm1_selected,
    MAIN_EXPERIMENTS["cm1"]["n_true"],
    EXPORT_DIR / "cm1_selected.csv",
)

#### Global-SE threshold

#### Number of selected variables

### CM2 — Mixed predictors, continuous response

The predictor vector contains 84 variables. The first 20 predictors are
independent $Bernoulli(0.2)$ variables, the next 20 are independent
$Bernoulli(0.5)$ variables, and the remaining 44 predictors follow a
multivariate normal distribution with unit marginal variance and pairwise
correlation 0.3.

The response is

$$
Y = f(X) + \varepsilon,
\qquad
\varepsilon \sim N(0,\sigma^2),
$$

with

$$
f(x)
=
-4 + x_1
+ \sin(\pi x_1x_{44})
- x_{21}
+ 0.6x_{41}x_{42}
- \exp[-2(x_{42}+1)^2]
- x_{43}^2
+ 0.5x_{44}.
$$

The experiment uses $p=84$, $n\in\{500,1000\}$, and
$\sigma^2\in\{1,10\}$.

In [15]:
cm2_summary, cm2_selected = aggregate_experiment_results(
    main_results["cm2"]
)

display_summary_table(
    cm2_summary,
    "global_se",
    EXPORT_DIR / "cm2_summary.csv",
)

display_n_selected_summary(
    cm2_selected,
    MAIN_EXPERIMENTS["cm2"]["n_true"],
    EXPORT_DIR / "cm2_selected.csv",
)

#### Global-SE threshold

#### Number of selected variables

### BM1 - Mixed predictors, binary response

In the first scenario with a binary response, we sample predictors $x_1, \dots, x_{\lceil p/2 \rceil} \sim Bernoulli(0.5)$ independently and $x_{\lceil p/2 \rceil + 1}, \dots, x_p \sim Unif(0, 1)$ independently. We sample the response $y$ from $Bernoulli(\Phi(f(x)))$ where $f$ is the same as in the first mixed-continuous scenario.

In [16]:
bm1_summary, bm1_selected = aggregate_experiment_results(
    main_results["bm1"]
)

display_summary_table(
    bm1_summary,
    "global_se",
    EXPORT_DIR / "bm1_summary.csv",
)

display_n_selected_summary(
    bm1_selected,
    MAIN_EXPERIMENTS["bm1"]["n_true"],
    EXPORT_DIR / "bm1_selected.csv",
)

#### Global-SE threshold

#### Number of selected variables

### BM2 - Mixed predictors, binary response

In this scenario, we sample predictors $x_1, \dots, x_{20} \sim Bernoulli(0.2)$, $x_{21}, \dots, x_{40} \sim Bernoulli(0.5)$ and $x_{41}, \dots, x_{84}$ from a multivariate normal distribution with mean 0, variance 1 and correlation 0.3 independently and response $y$ from  $Bernoulli(\Phi(f(x)))$, where $f$ is the same as in the second mixed-continuous scenario.

In [17]:
bm2_summary, bm2_selected = aggregate_experiment_results(
    main_results["bm2"]
)

display_summary_table(
    bm2_summary,
    "global_se",
    EXPORT_DIR / "bm2_summary.csv",
)

display_n_selected_summary(
    bm2_selected,
    MAIN_EXPERIMENTS["bm2"]["n_true"],
    EXPORT_DIR / "bm2_selected.csv",
)

#### Global-SE threshold

#### Number of selected variables

## Hyperparameter sensitivity

The sensitivity experiments examine whether the relative behavior of standard
and logML-weighted variable selection persists when selected BART prior
hyperparameters are changed.

The reference configuration is

$$
m=20,\qquad \alpha=0.95,\qquad \beta=2,\qquad k=2.
$$

One of $m$, $\beta$, or $k$ is varied at a time:

- $m\in\{10,20,50\}$,
- $\beta\in\{1,2,3\}$,
- $k\in\{1,2,3\}$.

The middle value in each set is the reference configuration. The tree-depth
base parameter $\alpha=0.95$ is held fixed.

Each simulation repeat is paired across hyperparameter configurations: the same
simulated dataset and repeat-level random seed are used for the reference and
all six alternative configurations.

Three summaries are considered for each scenario:

1. **Performance by configuration** reports the absolute feature-selection
   performance at each setting.
2. **Paired change from the reference** reports signed within-repeat changes,
   preserving whether a metric increased or decreased.
3. **Sensitivity magnitude** reports mean absolute within-repeat changes,
   together with Jaccard distance between the selected-variable sets.

In [18]:
configuration_table = pd.DataFrame(
    [
        {
            "Configuration": config_id,
            "Varied parameter": values["varied_parameter"],
            "m": values["m"],
            "alpha": values["alpha"],
            "beta": values["beta"],
            "k": values["k"],
        }
        for config_id, values
        in EXPECTED_HYPERPARAMETER_CONFIGS.items()
    ]
)

configuration_table["Varied parameter"] = (
    configuration_table["Varied parameter"]
    .replace({"reference": "—"})
)

configuration_table

,Configuration,Varied parameter,m,alpha,beta,k
0,reference,—,20,0.95,2.0,2.0
1,m_low,m,10,0.95,2.0,2.0
2,m_high,m,50,0.95,2.0,2.0
3,beta_low,beta,20,0.95,1.0,2.0
4,beta_high,beta,20,0.95,3.0,2.0
5,k_low,k,20,0.95,2.0,1.0
6,k_high,k,20,0.95,2.0,3.0


### Continuous predictors scenario - CC1

The sensitivity study uses Scenario CC1 with $n=500$, $p=50$,
$\sigma^2=1$, and 50 paired simulation repeats.

This provides a relatively favorable continuous-response setting in which the
sensitivity of the two importance measures can be compared without the severe
performance floor seen in the binary-response experiment.

In [19]:
cc1_hyper = sensitivity_results["cc1"]

cc1_hyper_summary = aggregate_hyperparameter_results(
    cc1_hyper
)

cc1_paired = aggregate_paired_hyperparameter_results(
    cc1_hyper
)

cc1_sensitivity = aggregate_hyperparameter_sensitivity(
    cc1_hyper
)

#### Performance by configuration

Absolute variable-selection performance under each hyperparameter
configuration. The reference setting appears once within each parameter block
so that the alternative values can be compared directly with it.

In [20]:
display_hyperparameter_summary(
    cc1_hyper_summary,
    "global_se",
    EXPORT_DIR / "cont_summary.csv",
)

#### Paired change from the reference

For each repeat, the alternative configuration is compared with the reference
configuration fitted to the same simulated dataset.

Positive values indicate an increase relative to the reference configuration;
negative values indicate a decrease. Jaccard distance measures turnover in the
selected-variable set, with 0 indicating identical sets.

In [21]:
display_paired_summary(
    cc1_paired,
    "global_se",
    EXPORT_DIR / "cont_paired.csv",
)

#### Sensitivity magnitude

Mean absolute within-repeat changes measure the magnitude of sensitivity
without allowing increases and decreases to cancel across repeats.

Jaccard distance is reported separately because stability of aggregate
performance statistics does not necessarily imply stability of the selected
feature set.

In [22]:
display_sensitivity_comparison(
    cc1_sensitivity,
    "global_se",
    output_path=EXPORT_DIR / "cont_sensitivity.csv",
)

*Values are mean absolute changes from the reference configuration; parentheses contain SEs. Jaccard distance measures change in the selected-variable set, with 0 indicating identical sets.*

### Mixed predictors scenario - CM2

The sensitivity study uses Scenario CM2 with $n=500$, $p=84$,
$\sigma^2=10$, and 50 paired simulation repeats.

This is a more difficult continuous-response setting with mixed predictor
types, correlated continuous predictors, and six active variables.

In [23]:
cm2_hyper = sensitivity_results["cm2"]

cm2_hyper_summary = aggregate_hyperparameter_results(
    cm2_hyper
)

cm2_paired = aggregate_paired_hyperparameter_results(
    cm2_hyper
)

cm2_sensitivity = aggregate_hyperparameter_sensitivity(
    cm2_hyper
)

#### Performance by configuration

Absolute variable-selection performance under each hyperparameter
configuration. The reference setting appears once within each parameter block
so that the alternative values can be compared directly with it.

In [24]:
display_hyperparameter_summary(
    cm2_hyper_summary,
    "global_se",
    EXPORT_DIR / "mix_summary.csv",
)

#### Paired change from the reference

For each repeat, the alternative configuration is compared with the reference
configuration fitted to the same simulated dataset.

Positive values indicate an increase relative to the reference configuration;
negative values indicate a decrease. Jaccard distance measures turnover in the
selected-variable set, with 0 indicating identical sets.

In [25]:
display_paired_summary(
    cm2_paired,
    "global_se",
    EXPORT_DIR / "mix_paired.csv",
)

#### Sensitivity magnitude

Mean absolute within-repeat changes measure the magnitude of sensitivity
without allowing increases and decreases to cancel across repeats.

Jaccard distance is reported separately because stability of aggregate
performance statistics does not necessarily imply stability of the selected
feature set.

In [26]:
display_sensitivity_comparison(
    cm2_sensitivity,
    "global_se",
    output_path=EXPORT_DIR / "mix_sensitivity.csv",
)

*Values are mean absolute changes from the reference configuration; parentheses contain SEs. Jaccard distance measures change in the selected-variable set, with 0 indicating identical sets.*

### Binary-response scenario - BM1

The sensitivity study uses Scenario BM1 with $n=1000$, $p=50$,
$\sigma^2=1$, and 50 paired simulation repeats.

Both methods perform poorly in this scenario, so changes in precision, recall,
and F1 need to be interpreted with care. In particular, apparent stability near
a performance floor does not imply that the same variables are being selected.

In [27]:
bm1_hyper = sensitivity_results["bm1"]

bm1_hyper_summary = aggregate_hyperparameter_results(
    bm1_hyper
)

bm1_paired = aggregate_paired_hyperparameter_results(
    bm1_hyper
)

bm1_sensitivity = aggregate_hyperparameter_sensitivity(
    bm1_hyper
)

#### Performance by configuration

Absolute variable-selection performance under each hyperparameter
configuration. The reference setting appears once within each parameter block
so that the alternative values can be compared directly with it.

In [28]:
display_hyperparameter_summary(
    bm1_hyper_summary,
    "global_se",
    EXPORT_DIR / "bin_summary.csv",
)

#### Paired change from the reference

For each repeat, the alternative configuration is compared with the reference
configuration fitted to the same simulated dataset.

Positive values indicate an increase relative to the reference configuration;
negative values indicate a decrease. Jaccard distance measures turnover in the
selected-variable set, with 0 indicating identical sets.

In [29]:
display_paired_summary(
    bm1_paired,
    "global_se",
    EXPORT_DIR / "bin_paired.csv",
)

#### Sensitivity magnitude

Mean absolute within-repeat changes measure the magnitude of sensitivity
without allowing increases and decreases to cancel across repeats.

Jaccard distance is reported separately because stability of aggregate
performance statistics does not necessarily imply stability of the selected
feature set.

In [30]:
display_sensitivity_comparison(
    bm1_sensitivity,
    "global_se",
    output_path=EXPORT_DIR / "bin_sensitivity.csv",
)

*Values are mean absolute changes from the reference configuration; parentheses contain SEs. Jaccard distance measures change in the selected-variable set, with 0 indicating identical sets.*

### Optional report figures

`plot_hyperparameter_sensitivity()` is retained in
`sensitivity_analysis_helpers.py` because the resulting figures are used in
the report. They are not rendered by default here in order to keep this
analysis notebook compact.

In [31]:
SHOW_REPORT_FIGURES = False

if SHOW_REPORT_FIGURES:
    plot_hyperparameter_sensitivity(
        cc1_sensitivity,
        "global_se",
    )

    plot_hyperparameter_sensitivity(
        cm2_sensitivity,
        "global_se",
    )

    plot_hyperparameter_sensitivity(
        bm1_sensitivity,
        "global_se",
    )